# 01 — Data understanding

CIC-IDS2017 was captured over five working days in July 2017. Monday is benign traffic only; attacks were staged Tuesday through Friday. Eight CSV files, 80 columns each, 2,830,743 rows in total.

**Pipeline module:** `src/data_loader.py`

*Every number and figure below was produced by the pipeline in `src/`. This
notebook reads those results; it does not re-implement them.*

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

def table(name, **kw):
    """Read one of the pipeline's result tables."""
    return pd.read_csv(TABLES / name, **kw)

def figure(name):
    """Display one of the pipeline's figures."""
    return Image(filename=str(FIGURES / name))

print(f"project root: {ROOT}")

project root: C:\Users\babu\OneDrive\Desktop\ML Project


## The capture files

Each file is one period of traffic. The misspelling in `Infilteration` is the
dataset authors' own — the filenames are left exactly as published.

In [2]:
files = pd.DataFrame([
    ("Monday-WorkingHours",               529_918, "Benign only"),
    ("Tuesday-WorkingHours",              445_909, "FTP / SSH brute force"),
    ("Wednesday-workingHours",            692_703, "DoS variants, Heartbleed"),
    ("Thursday-Morning-WebAttacks",       170_366, "XSS, SQL injection, brute force"),
    ("Thursday-Afternoon-Infilteration",  288_602, "Infiltration"),
    ("Friday-Morning",                    191_033, "Botnet"),
    ("Friday-Afternoon-PortScan",         286_467, "Port scanning"),
    ("Friday-Afternoon-DDos",             225_745, "DDoS"),
], columns=["file", "rows", "contains"])

files.loc[len(files)] = ("TOTAL", files["rows"].sum(), "")
files

,file,rows,contains
0,Monday-WorkingHours,529918,Benign only
1,Tuesday-WorkingHours,445909,FTP / SSH brute force
2,Wednesday-workingHours,692703,"DoS variants, Heartbleed"
3,Thursday-Morning-WebAttacks,170366,"XSS, SQL injection, brute force"
4,Thursday-Afternoon-Infilteration,288602,Infiltration
5,Friday-Morning,191033,Botnet
6,Friday-Afternoon-PortScan,286467,Port scanning
7,Friday-Afternoon-DDos,225745,DDoS
8,TOTAL,2830743,


## Memory: the first real constraint

This project was built on a machine with 7.8 GB of RAM. Loaded naively the merged
frame needs roughly 1.7 GB, Windows takes about 3 GB, and pandas makes copies during
a concatenation — so a straightforward merge exhausts memory.

`data_loader.py` loads and shrinks one file at a time, downcasting `float64 → float32`
and storing labels as categories. The merged dataset then occupies **686 MB**.

In [3]:
import inspect
import data_loader

print(inspect.getsource(data_loader.downcast_numeric))

def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """
    Shrink float64 -> float32 and int64 -> the smallest safe integer type.

    This roughly halves memory use. This project runs on a machine with about
    8 GB of RAM and the merged dataset is ~2.8 million rows, so without this
    step the merge alone would exhaust memory. float32 keeps roughly 7
    significant digits, far more precision than these flow statistics carry.
    """
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df



## Running the profile

With the eight CSVs in `data/`, this reproduces the full profile — row and column
counts, dtypes, missing values, infinities, duplicates and the class distribution:

```bash
python src/data_loader.py
```

It takes a few minutes on the full 2.83 million rows, so the resulting class
distribution is read below rather than recomputed.

In [4]:
raw_counts = table("class_distribution.csv", index_col=0)
raw_counts["percent"] = (raw_counts["count"] / raw_counts["count"].sum() * 100).round(4)
raw_counts

,count,percent
Label,,
BENIGN,2273097,80.3004
DoS Hulk,231073,8.1630
PortScan,158930,5.6144
DDoS,128027,4.5227
DoS GoldenEye,10293,0.3636
FTP-Patator,7938,0.2804
SSH-Patator,5897,0.2083
DoS slowloris,5796,0.2048
DoS Slowhttptest,5499,0.1943


## What the profile found

Four problems in the published data, each handled in notebook 03:

| Problem | Extent |
|---|---|
| Corrupted label text | 2,180 rows — `Web Attack` labels contain bytes `EF BF BD` where an en-dash belongs |
| Infinite values | `Flow Bytes/s` and `Flow Packets/s` divide by a zero-length flow duration |
| Duplicate rows | 11.66% of the dataset |
| Zero-variance columns | 8 columns CICFlowMeter never populated |

The class imbalance is the defining property of this dataset: **BENIGN is 80.3% of
raw rows, and Heartbleed has 11 rows in total** — a ratio of roughly 206,000:1.